In [ ]:
import os
from openai import OpenAI
from openai_fused_model import FusedClient, RoutingStrategy, ModelConfig

os.environ["OPENAI_API_KEY"] = "key"


In [31]:
# 1. Standart OpenAI istemcilerinizi oluşturun
# Farklı API anahtarları, farklı sağlayıcılar (örn. DeepSeek, Groq) veya farklı base_url'ler kullanabilirsiniz.
client_primary = OpenAI() 
client_secondary = OpenAI()
client_backup = OpenAI()

# 2. Bu istemcileri yönetecek FusedClient'ı oluşturun
# İstemciler için öncelik ve hız limitleri de tanımlayabilirsiniz
configs = [
    # Birincil: Yüksek öncelik. Belirli bir model (örn: gpt-4) için atanabilir.
    ModelConfig(priority=100, max_rpm=2, model="gpt-4.1"), 
    # Yedek: Düşük öncelik, yüksek hız limiti. Model belirtilmezse genel amaçlı (fallback) veya model farketmeksizin kullanılabilir.
    ModelConfig(priority=50, max_rpm=2, model="gpt-4.1-mini"),
    ModelConfig(priority=25, model="gpt-4.1-nano")
]

fused_client = FusedClient(
    clients=[client_primary, client_secondary, client_backup],
    model_configs=configs,
    strategy=RoutingStrategy.COST_AWARE, # Önce yüksek öncelikli olanı dene
)

print("FusedClient hazır. İstek gönderiliyor...")

FusedClient hazır. İstek gönderiliyor...


In [34]:
# 3. Standart OpenAI formatında kullanın
try:
    response = fused_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Just say hello."}
        ]
    )
    
    # Cevabı yazdır (NOT: Geçersiz API keyler nedeniyle bu kod çalıştırılırsa hata verecektir)
    print(response.model_dump()["model"])

except Exception as e:
    print(f"İşlem sırasında hata oluştu (Beklenen durum, çünkü API keyler geçersiz): {e}")


gpt-4.1-mini-2025-04-14
